# Gradient Boosting Regressor
기울기 벡터 부스팅 회귀라고도 부를 수 있는 이 모델은 여러개의 작언 결정트리를 순서대로 학습시켜 예측 성능을 올리는 회귀 모델입니다.

**랜덤 포레스트**와 비슷하게 트리 여러개를 쓰지만 방식이 다릅니다.

랜덤 포레스트는 각자 따로 학습하여 투표 또는 평균을 내는 방식이지만 **Gradient Boosting**는 트리를 순서대로 학습시키게 됩니다.

예를 들어 1번 트리를 먼저 학습시킨 뒤, 2번 트리에서 1번 트리가 틀린 오차를 보정하는 방식을 반복하는 방식입니다.

# 특징
**Gradient Boosting**는 보통 성능이 꽤 잘 나오는 편입니다. 이는 랜덤 포레스트보다 더 정교한 특성간 상호작용 및 비선형적 관계를 잡을 수 있습니다.

예를 들면 무게가 무거우면서 마력이 높은 차, 특정 조건 조합에서 연비가 급격히 낮아지는 패턴 등을 찾을 수 있습니다.

또한 **Gradient Boosting**는 랜덤 포레스트보다 정교한만큼 튜닝에 민감한 모습을 보입니다. 특히 `learning_rate`를 통해 한 트리가 반영되는 정도를 설정할 수 있습니다. 그리고 대체로 얕은(`2`~`4`) 트리를 많이 쌓는 경우가 더 많습니다.

In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

data = fetch_ucirepo(id=9)

df = pd.concat((
  data.data.features,
  data.data.targets
), axis=1)
df

,displacement,cylinders,horsepower,weight,acceleration,model_year,origin,mpg
0,307.0,8,130.0,3504,12.0,70,1,18.0
1,350.0,8,165.0,3693,11.5,70,1,15.0
2,318.0,8,150.0,3436,11.0,70,1,18.0
3,304.0,8,150.0,3433,12.0,70,1,16.0
4,302.0,8,140.0,3449,10.5,70,1,17.0
...,...,...,...,...,...,...,...,...
393,140.0,4,86.0,2790,15.6,82,1,27.0
394,97.0,4,52.0,2130,24.6,82,2,44.0
395,135.0,4,84.0,2295,11.6,82,1,32.0
396,120.0,4,79.0,2625,18.6,82,1,28.0


In [2]:
df = df.dropna(axis=0)
X = df.drop("mpg", axis=1)
y = df["mpg"]

In [3]:
X = pd.get_dummies(X, columns=["origin"])
X.columns

Index(['displacement', 'cylinders', 'horsepower', 'weight', 'acceleration',
       'model_year', 'origin_1', 'origin_2', 'origin_3'],
      dtype='str')

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
  X, y,
  test_size=0.2,
  random_state=42
)

In [5]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

In [ ]:
# 파이프라인 = 전처리 + 모델을 하나로 묶어주는 도구
# 리스트안에는 그냥 ("이름", "부를 수 있는 객체")를 주는 형식
models = {
  "Linear Regression": Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
  ]),

  "Ridge": Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
  ]),

  "Random Forest": RandomForestRegressor(
    random_state=42
  ),

  "Gradient Boosting": GradientBoostingRegressor(
    random_state=42
  )
}

In [7]:
scoring = {
  "mae": "neg_mean_absolute_error",
  "rmse": "neg_root_mean_squared_error",
  "r2": "r2"
}

In [ ]:
results = []

for name, model in models.items():
  # 훈련 데이터셋을 5등분 해서 train:test = 4:1로 5번 테스트를 진행
  scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=5,
    scoring=scoring
  )

  results.append({
    "model": name,
    "MAE": -scores["test_mae"].mean(),
    "RMSE": -scores["test_rmse"].mean(),
    "R2": scores["test_r2"].mean()
  })

results_df = pd.DataFrame(results)
results_df
# 결과적으로 그레디언트 부스팅이 가장 점수가 높게 나옴

,model,MAE,RMSE,R2
0,Linear Regression,2.665324,3.419213,0.810337
1,Ridge,2.657016,3.415776,0.810815
2,Random Forest,2.129383,2.970974,0.856003
3,Gradient Boosting,2.089518,2.887986,0.862763


In [10]:
best_model = GradientBoostingRegressor(random_state=42)

best_model.fit(X_train, y_train)

pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

mae, float(rmse), r2

(1.8040775712119654, 2.557079958548544, 0.8718929624357508)